# Handwritten Digit Recognition with CNN
**Deep Learning Lab | MNIST | TensorFlow/Keras**

Train in Google Colab; export the model for a separate Streamlit application.

## 1. Imports and reproducibility

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

tf.keras.utils.set_random_seed(42)
print("TensorFlow:", tf.__version__)

## 2. Load and explore MNIST
Images are grayscale (28×28), with labels from 0 to 9.

In [ ]:
(x_train_raw, y_train), (x_test_raw, y_test) = tf.keras.datasets.mnist.load_data()
print("Train:", x_train_raw.shape, y_train.shape)
print("Test:", x_test_raw.shape, y_test.shape)
fig, ax = plt.subplots(2, 5, figsize=(10, 5))
for a, i in zip(ax.ravel(), range(10)):
    a.imshow(x_train_raw[i], cmap="gray")
    a.set_title(f"Label: {y_train[i]}")
    a.axis("off")
plt.tight_layout(); plt.show()

## 3. Preprocessing
Normalize the pixel range from 0–255 to 0–1 and add a grayscale channel.

In [ ]:
x_train = (x_train_raw.astype("float32") / 255.0)[..., None]
x_test = (x_test_raw.astype("float32") / 255.0)[..., None]
print(x_train.shape, x_test.shape)

## 4. Build the CNN
Convolution extracts local patterns; max pooling reduces feature maps; the final softmax produces 10 class scores.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),
    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Conv2D(64, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(10, activation="softmax")
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 5. Train the model
Use a validation split inside the training data; keep the test set untouched until evaluation.

In [ ]:
history = model.fit(x_train, y_train, epochs=5, batch_size=128, validation_split=0.1, verbose=2)

## 6. Learning curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric in zip(axes, ["accuracy", "loss"]):
    ax.plot(history.history[metric], label="train")
    ax.plot(history.history["val_" + metric], label="validation")
    ax.set_title(metric.title()); ax.set_xlabel("Epoch"); ax.legend(); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

## 7. Evaluate on the test set

In [ ]:
loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {accuracy:.2%} | Test loss: {loss:.4f}")
proba = model.predict(x_test, verbose=0)
y_pred = proba.argmax(axis=1)
print(classification_report(y_test, y_pred))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues", values_format="d")
plt.title("MNIST test set confusion matrix"); plt.show()

## 8. Visualize mistakes

In [ ]:
errors = np.where(y_pred != y_test)[0][:12]
fig, ax = plt.subplots(3, 4, figsize=(10, 7))
for a, i in zip(ax.ravel(), errors):
    a.imshow(x_test_raw[i], cmap="gray")
    a.set_title(f"True {y_test[i]} → Pred {y_pred[i]}")
    a.axis("off")
for a in ax.ravel()[len(errors):]: a.axis("off")
plt.tight_layout(); plt.show()

## 9. Export the CNN
Save as `mnist_cnn.keras` and place it under `model/` in the Streamlit project.

In [ ]:
model.save("mnist_cnn.keras")
print("Saved model to mnist_cnn.keras")
# Colab only: uncomment to download the file
# from google.colab import files
# files.download("mnist_cnn.keras")

## 10. Deploy and reflect
Run `streamlit run app.py` on a machine hosting the app, then open its URL on the tablet. Test the digits 0–9. Compare errors on tablet drawings with errors on MNIST.

**Question:** why do we crop, resize, and center the canvas image before the CNN sees it?